## Modeling the Effects of Temperature on Race Performance

This will seek to work with the same set as the prediction modeling for driver behavior, but uses the weather dataset for session data in the analysis.

In [71]:
# Install FastF1 if not already installed
# !pip install fastf1

import fastf1
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')

import os
os.makedirs('f1_cache', exist_ok=True)  # Add this line

fastf1.Cache.enable_cache('f1_cache')
# Enable local caching to avoid repeated API calls
fastf1.Cache.enable_cache('f1_cache')  # Creates a local folder called f1_cache

In [2]:
# Preview weather data availability
weather = session.weather_data
print(f"Weather data shape: {weather.shape}")
print(f"Weather columns: {list(weather.columns)}")
weather[['Time', 'AirTemp', 'TrackTemp', 'Humidity', 'Rainfall']].head()

Weather data shape: (157, 8)
Weather columns: ['Time', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed']


,Time,AirTemp,TrackTemp,Humidity,Rainfall
0,0 days 00:00:14.093000,18.9,26.5,46.0,False
1,0 days 00:01:14.084000,18.9,26.5,46.0,False
2,0 days 00:02:14.093000,18.9,26.5,46.0,False
3,0 days 00:03:14.090000,18.9,26.2,45.0,False
4,0 days 00:04:14.091000,18.9,26.2,46.0,False


In [62]:
races = ['United States', 'Bahrain', 'Saudi Arabia', 'Australia', 'Japan', 'China']
years = [2021, 2022, 2023, 2024, 2025]
all_data = []

for race in races:
    for year in years: 
        session = fastf1.get_session(year, race, 'R')
        session.load(telemetry=False, weather=True)
        
        laps = session.laps.copy()
        weather = session.weather_data.copy()
        
        laps['Race'] = race
        laps['Year'] = year
        
        laps = laps.sort_values('Time')
        weather = weather.sort_values('Time')
    
        laps_with_weather = pd.merge_asof(laps, weather, on='Time', direction='backward')
        
        all_data.append(laps_with_weather)

combined_df = pd.concat(all_data, ignore_index=True)

combined_df['LapTime_Seconds'] = combined_df['LapTime'].dt.total_seconds()
combined_df.columns

core           INFO 	Loading data for United States Grand Prix - Race [v3.8.0]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate', 'Race', 'Year', 'AirTemp', 'Humidity',
       'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed',
       'LapTime_Seconds'],
      dtype='object')

In [66]:
combined_df = combined_df[combined_df['PitInTime'].isna() & combined_df['PitOutTime'].isna()]
combined_df = combined_df[combined_df['Compound'] != 'None']
combined_df['Compound'].value_counts()

Compound
HARD            14924
MEDIUM           9125
SOFT             3235
INTERMEDIATE     1066
WET                56
Name: count, dtype: int64

# Temperature Multivariate Regression

**Model Specifications**: 
- Controlled for race and year fixed effects 
- Didn't control for each driver due to PMCL
- Lapnumber control needed due to fuel effect on race tine
- Windspeed and TrackTemp captures effect of wind direction and air temperature. This avoid multicolinearity
- Rainfall control which directly affects tire compound
- Used an interaction between track temperature and tire compound to capture effect of temperature on each compound, avoiding using a single compound as a baseline
- Tyrelife over the course of the race is a huge determinant that doesn't necessarily show whether the increased the lap time is from compound alone

In [69]:
# Multi-variate regression
reg = smf.ols(
    formula = '''LapTime_Seconds ~ 
                + C(Compound)
                + TrackTemp: C(Compound) - 1
                + TyreLife
                + WindSpeed
                + Rainfall
                + LapNumber
                + C(Year)
                + C(Race)
                ''',
              data = combined_df).fit(cov_type = 'HC2')

print(reg.summary())

                            OLS Regression Results                            
Dep. Variable:        LapTime_Seconds   R-squared:                       0.380
Model:                            OLS   Adj. R-squared:                  0.380
Method:                 Least Squares   F-statistic:                       nan
Date:                Tue, 21 Apr 2026   Prob (F-statistic):                nan
Time:                        17:12:58   Log-Likelihood:            -1.0293e+05
No. Observations:               27998   AIC:                         2.059e+05
Df Residuals:                   27975   BIC:                         2.061e+05
Df Model:                          22                                         
Covariance Type:                  HC2                                         
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
C(

## Analysis

**Temperature**: Windspeed, rainfall, and track temperature had statistial signficance

**Tire Compound**: all tire compounds except wet showed statistical signficance

- **Soft**: reduced lap times by 0.3351 seconds
- **Medium**: reduced lap times by 0.4052 seconds
- **Intermediate**: increased laptimes 3.6302 seconds
- **Hard**: reduced lapt times -0.4588 seconds

**Non-statistically Significant Factors**: the only variables were wet tire compounds